<a href="https://colab.research.google.com/github/kangwonlee/nmisp/blob/main/60_linear_algebra_2/212_Generalized_Eigenvalue_Vibration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 일반화 고유치 문제 : 2자유도 진동<br>Generalized Eigenvalue Problem : 2-DOF Vibration

지금까지의 고유치 문제는 $A\mathbf{x} = \lambda \mathbf{x}$ 형태였다. 많은 공학 문제는 그 대신 행렬이 **두 개** 나오는 **일반화 고유치 문제** $A\mathbf{x} = \lambda B\mathbf{x}$ 로 표현된다.<br>
The eigenvalue problems so far had the form $A\mathbf{x} = \lambda \mathbf{x}$. Many engineering problems instead lead to a **generalized eigenvalue problem** $A\mathbf{x} = \lambda B\mathbf{x}$, with **two** matrices.

대표적인 예가 기계 진동이다.<br>A classic example is mechanical vibration.


## 자유 진동에서 나오는 일반화 고유치 문제<br>The generalized eigenproblem of free vibration

질량행렬 $\mathbb{M}$ 과 강성행렬 $\mathbb{K}$ 를 가진 자유 진동계의 운동 방정식은 다음과 같다.<br>
A freely vibrating system with mass matrix $\mathbb{M}$ and stiffness matrix $\mathbb{K}$ obeys:

$$ \mathbb{M}\,\ddot{\mathbf{x}} + \mathbb{K}\,\mathbf{x} = \mathbf{0}. $$

조화 운동 $\mathbf{x}(t) = \boldsymbol{\phi}\, e^{i\omega t}$ 을 가정하면 $\ddot{\mathbf{x}} = -\omega^2 \mathbf{x}$ 이므로:<br>
Assuming harmonic motion $\mathbf{x}(t) = \boldsymbol{\phi}\, e^{i\omega t}$ gives $\ddot{\mathbf{x}} = -\omega^2 \mathbf{x}$, hence:

$$ \mathbb{K}\,\boldsymbol{\phi} = \omega^2\, \mathbb{M}\,\boldsymbol{\phi}. $$

이것이 바로 $A\mathbf{x} = \lambda B\mathbf{x}$ 꼴이다 — $A=\mathbb{K}$, $B=\mathbb{M}$, $\lambda = \omega^2$. 고유치는 **고유진동수의 제곱**, 고유벡터 $\boldsymbol{\phi}$ 는 **모드 형상(mode shape)** 이다.<br>
This is exactly $A\mathbf{x} = \lambda B\mathbf{x}$ with $A=\mathbb{K}$, $B=\mathbb{M}$, $\lambda = \omega^2$. The eigenvalues are the **squared natural frequencies** and the eigenvectors $\boldsymbol{\phi}$ are the **mode shapes**.


In [ ]:
# 행렬·수치 계산 / matrix and numerical features
import numpy as np
import numpy.linalg as nl
import scipy.linalg as sl
import matplotlib.pyplot as plt


## 2자유도 진동계<br>A 2-DOF vibration system

질량 $m_1=100$, $m_2=25$ (kg), 스프링 $k_1=k_2=36\,000$ (N/m). $\mathbb{M}$ 는 대각, $\mathbb{K}$ 는 대칭이다.<br>
Masses $m_1=100$, $m_2=25$ (kg); springs $k_1=k_2=36\,000$ (N/m). $\mathbb{M}$ is diagonal and $\mathbb{K}$ is symmetric.


In [ ]:
m1, m2 = 100.0, 25.0
k1 = k2 = 36e3

matM = np.array([
    [m1, 0.0],
    [0.0, m2],
])

matK = np.array([
    [k1 + k2, -k2],
    [   -k2,   k2],
])
matM, matK


In [ ]:
# 둘 다 대칭, 그리고 M 은 양의 정부호 / both symmetric, and M is positive-definite
assert np.allclose(matM, matM.T) and np.allclose(matK, matK.T)
assert np.all(nl.eigvalsh(matM) > 0)
print("M, K 대칭 / symmetric ; M 양의 정부호 / positive-definite")


## 순진한 방법 : $\mathbb{M}^{-1}\mathbb{K}$ 로 표준 문제 만들기<br>The naive route: reduce to a standard problem with $\mathbb{M}^{-1}\mathbb{K}$

$\mathbb{K}\boldsymbol{\phi} = \omega^2 \mathbb{M}\boldsymbol{\phi}$ 의 양변에 $\mathbb{M}^{-1}$ 을 곱하면 표준 고유치 문제 $(\mathbb{M}^{-1}\mathbb{K})\boldsymbol{\phi} = \omega^2 \boldsymbol{\phi}$ 가 된다.<br>
Multiplying $\mathbb{K}\boldsymbol{\phi} = \omega^2 \mathbb{M}\boldsymbol{\phi}$ by $\mathbb{M}^{-1}$ turns it into a standard problem $(\mathbb{M}^{-1}\mathbb{K})\boldsymbol{\phi} = \omega^2 \boldsymbol{\phi}$.


In [ ]:
matC = nl.inv(matM) @ matK
lam_naive = np.sort(nl.eigvals(matC).real)
lam_naive  # = omega^2


숫자는 맞다. 그러나 함정이 있다 — $\mathbb{M}^{-1}\mathbb{K}$ 는 더 이상 **대칭이 아니다**.<br>
The numbers are right. But there is a trap — $\mathbb{M}^{-1}\mathbb{K}$ is no longer **symmetric**.


In [ ]:
print("M^-1 K =\n", matC)
print("대칭인가? / symmetric?", np.allclose(matC, matC.T))


대칭을 잃으면, 대칭 문제가 **보장**해 주던 것들 — 실수 고유치와 서로 직교인 모드 — 도 일반적으로 보장되지 않는다. 이 작은 예에서는 우연히 괜찮지만, 좋은 습관이 아니다. 대칭 구조를 **그대로 살리는** 도구를 쓰자.<br>
Losing symmetry means losing the **guarantees** a symmetric problem gives — real eigenvalues and mutually orthogonal modes. It happens to be fine for this tiny case, but it is a bad habit. Use a tool that **keeps** the symmetric structure.


## 올바른 도구 : `scipy.linalg.eigh(K, M)`<br>The right tool: `scipy.linalg.eigh(K, M)`

`eigh` 는 $A$ 가 대칭이고 $B$ 가 양의 정부호인 일반화 문제를 콜레스키 분해로 풀어, 대칭성을 유지한 채 **실수** 고유치(오름차순)와 $\mathbb{M}$-직교 모드를 돌려준다.<br>
`eigh` solves the generalized problem for symmetric $A$ and positive-definite $B$ via Cholesky, keeping symmetry — it returns **real** eigenvalues (ascending) and $\mathbb{M}$-orthogonal modes.


In [ ]:
lam, Phi = sl.eigh(matK, matM)   # K phi = lam M phi,  lam = omega^2
lam


In [ ]:
# 고유진동수 / natural frequencies
omega = np.sqrt(lam)        # rad/s
freq_hz = omega / (2 * np.pi)   # Hz
print("omega (rad/s):", omega)
print("f     (Hz)  :", freq_hz)


## 모드 형상<br>Mode shapes

각 열 $\boldsymbol{\phi}_i$ 가 한 모드의 형상이다. 1차 모드는 두 질량이 **같은 방향**으로 (동위상), 2차 모드는 **반대 방향**으로 (역위상) 움직인다.<br>
Each column $\boldsymbol{\phi}_i$ is one mode shape. In mode 1 the two masses move in the **same** direction (in phase); in mode 2 they move in **opposite** directions (out of phase).


In [ ]:
Phi


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharey=True)
dof = [1, 2]   # 질량 번호 / mass index
for i, ax in enumerate(axes):
    ax.plot([0, 0], [1, 2], "o--", color="gray")          # 평형 위치 / equilibrium
    ax.plot(Phi[:, i], dof, "o-", color="C0")              # 변형 형상 / deflected shape
    ax.set_title(f"mode {i}: f = {freq_hz[i]:.2f} Hz")
    ax.set_yticks(dof)
    ax.set_xlabel("displacement")
    ax.axvline(0, color="k", lw=0.5)
axes[0].set_ylabel("mass")
plt.tight_layout()
plt.show()


## 모드의 $\mathbb{M}$-직교성<br>$\mathbb{M}$-orthogonality of the modes

대칭-정부호 문제이므로 모드는 $\mathbb{M}$ 에 대해 직교한다. `eigh` 는 $\Phi^{T}\mathbb{M}\,\Phi = I$ 가 되도록 정규화해 준다.<br>
For a symmetric-definite problem the modes are orthogonal with respect to $\mathbb{M}$. `eigh` normalizes them so that $\Phi^{T}\mathbb{M}\,\Phi = I$.


In [ ]:
PtMP = Phi.T @ matM @ Phi
print(np.round(PtMP, 9))
assert np.allclose(PtMP, np.eye(2)), "modes are not M-orthonormal"
print("Phi^T M Phi = I  확인 / verified")


## 이어지는 이야기<br>Where this leads

- 여기서 구한 고유진동수는 같은 2자유도 계의 **시간 영역 진동 시뮬레이션** 에서 그대로 나타난다.<br>
  The natural frequencies found here are exactly what the **time-domain vibration simulation** of the same 2-DOF system exhibits.

- $B = \mathbb{M} = I$ 이면 일반화 문제는 표준 고유치 문제로 되돌아가고, 그것은 **QR 알고리듬** 으로 푼다.<br>
  When $B = \mathbb{M} = I$ the generalized problem collapses to the standard one, solved by the **QR algorithm**.

- $\mathbb{M}$ 이 대칭-정부호가 아니거나 (예: 질량이 없는 자유도) 일반(비대칭) 행렬 쌍이면 `eigh` 를 쓸 수 없다. 그때 쓰는 것이 **QZ 알고리듬** 이다.<br>
  When $\mathbb{M}$ is not symmetric-positive-definite (e.g. a massless degree of freedom) or the pair is a general non-symmetric one, `eigh` no longer applies — that is where the **QZ algorithm** comes in.


## Final Bell<br>마지막 종


In [ ]:
# stackoverfow.com/a/24634221
import os
os.system("printf '\a'");
